In [1]:
from pyspark.sql import SparkSession 
from pyspark.sql.functions import col, lit

In [2]:
spark = SparkSession.builder.appName("Phase2").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/23 09:15:46 WARN Utils: Your hostname, LAPTOP-H1TA7CV3, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/23 09:15:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/23 09:15:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/23 09:15:48 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
data_combatants = [
    ("Ser Duncan the Tall", "Hedge Knight", 2),
    ("Prince Aerion Brightflame", "Royalty", 15),
    ("Ser Lyonel Baratheon", "Nobility", 12),
    ("Ser Robyn Rhysling", "Nobility", 8)
]
df_combatants = spark.createDataFrame(data_combatants, ["name", "status", "tourneys_won"])

---
### 1. Basic Operations: Structuring the Roster
These are your bread-and-butter narrow transformations.

 - select(): Chooses specific columns.

 - filter(): Subsets rows based on conditions (identical to a SQL WHERE clause).

 - withColumn(): Creates a new column or overwrites an existing one.

 - drop(): Removes a column.

---

In [4]:
df_cleaned = df_combatants \
            .filter(col("status") != "Royalty") \
            .withColumn("is_vaterian", col("tourneys_won") > 5) \
            .withColumn("current_location", lit("Ashford Meadow")) \
            .drop("Tourneys_won")

print("Non-Royal Combatants and Veteran Status:")
df_cleaned.show(truncate=False)

Non-Royal Combatants and Veteran Status:


+--------------------+------------+-----------+----------------+
|name                |status      |is_vaterian|current_location|
+--------------------+------------+-----------+----------------+
|Ser Duncan the Tall |Hedge Knight|false      |Ashford Meadow  |
|Ser Lyonel Baratheon|Nobility    |true       |Ashford Meadow  |
|Ser Robyn Rhysling  |Nobility    |true       |Ashford Meadow  |
+--------------------+------------+-----------+----------------+



### 2. Aggregations (Wide Transformations)
When you need to summarize data across the entire cluster, you use aggregations. This triggers a shuffle, much like a GROUP BY in PostgreSQL. Spark groups the data by a specific key, moving records across Executor nodes so that all matching keys end up on the same machine before computing the count(), sum(), or avg().

In [5]:
df_stats = df_combatants.groupBy("status").avg("tourneys_won").withColumnRenamed("avg(tourneys_won)", "avg_tourneys_won")
df_stats.show(truncate=False)

+------------+----------------+
|status      |avg_tourneys_won|
+------------+----------------+
|Hedge Knight|2.0             |
|Royalty     |15.0            |
|Nobility    |10.0            |
+------------+----------------+



### 3. Joins: Merging the Camps
PySpark supports all the standard SQL joins (inner, left, right, outer). Because data is distributed, joining two large DataFrames requires a massive shuffle—Spark has to align the matching keys from different nodes across the network.

In [8]:
data_squires = [
    ("Ser Duncan the Tall", "Aegon (Egg)"),
    ("Ser Lyonel Baratheon", "Pate")
]
df_squires = spark.createDataFrame(data_squires, ["knight_name", "squire_name"])

df_camps = df_combatants.join(
    df_squires,
    df_combatants.name == df_squires.knight_name,
    how="left"
)
print("Ashford Camp")
display(df_camps.show(truncate=False))
print("If a column dropped")
df_camps.drop("knight_name").show(truncate=False)

Ashford Camp
+-------------------------+------------+------------+--------------------+-----------+
|name                     |status      |tourneys_won|knight_name         |squire_name|
+-------------------------+------------+------------+--------------------+-----------+
|Ser Duncan the Tall      |Hedge Knight|2           |Ser Duncan the Tall |Aegon (Egg)|
|Prince Aerion Brightflame|Royalty     |15          |NULL                |NULL       |
|Ser Lyonel Baratheon     |Nobility    |12          |Ser Lyonel Baratheon|Pate       |
|Ser Robyn Rhysling       |Nobility    |8           |NULL                |NULL       |
+-------------------------+------------+------------+--------------------+-----------+



None

If a column dropped
+-------------------------+------------+------------+-----------+
|name                     |status      |tourneys_won|squire_name|
+-------------------------+------------+------------+-----------+
|Ser Duncan the Tall      |Hedge Knight|2           |Aegon (Egg)|
|Prince Aerion Brightflame|Royalty     |15          |NULL       |
|Ser Lyonel Baratheon     |Nobility    |12          |Pate       |
|Ser Robyn Rhysling       |Nobility    |8           |NULL       |
+-------------------------+------------+------------+-----------+



### 4. Window Functions: The Analytical Edge
Window functions allow you to perform calculations across a specific subset of rows (a "window") without collapsing them into a single output row like groupBy() does. If you've used OVER (PARTITION BY ... ORDER BY ...) in PostgreSQL, this is the exact same mental model.

Example: Dunk wants to know his rank compared only to other Hedge Knights, while Prince Aerion is ranked against other Royalty.

In [10]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank

# 1. Define the Window partition and ordering
# We partition by status (putting royalty with royalty) and order by wins descending
window_spec = Window.partitionBy("status").orderBy(col("tourneys_won").desc())

# 2. Apply a window function (rank) over the specification
df_ranked = df_combatants.withColumn("rank_in_class", rank().over(window_spec))

print("Combatants Ranked Within Their Social Class:")
df_ranked.show(truncate=False)

Combatants Ranked Within Their Social Class:
+-------------------------+------------+------------+-------------+
|name                     |status      |tourneys_won|rank_in_class|
+-------------------------+------------+------------+-------------+
|Ser Duncan the Tall      |Hedge Knight|2           |1            |
|Ser Lyonel Baratheon     |Nobility    |12          |1            |
|Ser Robyn Rhysling       |Nobility    |8           |2            |
|Prince Aerion Brightflame|Royalty     |15          |1            |
+-------------------------+------------+------------+-------------+



In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import col, sum

spark = SparkSession.builder.appName("AshfordBouts").getOrCreate()

data_bouts = [
    ("Ser Duncan", 1, 2),
    ("Ser Duncan", 2, 4),
    ("Ser Duncan", 3, 3),
    ("Prince Aerion", 1, 5),
    ("Prince Aerion", 2, 1),
    ("Prince Aerion", 3, 6)
]
df_bouts = spark.createDataFrame(data_bouts, ["combatant", "day", "bouts_fought"])

# TODO: Define your window_spec here
# TODO: Create df_cumulative_bouts using withColumn here

window_spec = Window.partitionBy("combatant").orderBy("day")
df_cumulative_bouts = df_bouts.withColumn("cumulative_bouts", sum("bouts_fought").over(window_spec))

df_cumulative_bouts.show()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/23 15:24:37 WARN Utils: Your hostname, LAPTOP-H1TA7CV3, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/23 15:24:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/23 15:24:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+-------------+---+------------+----------------+
|    combatant|day|bouts_fought|cumulative_bouts|
+-------------+---+------------+----------------+
|Prince Aerion|  1|           5|               5|
|Prince Aerion|  2|           1|               6|
|Prince Aerion|  3|           6|              12|
|   Ser Duncan|  1|           2|               2|
|   Ser Duncan|  2|           4|               6|
|   Ser Duncan|  3|           3|               9|
+-------------+---+------------+----------------+

